# Section 6: Multi-Graph & Polymorphism (Q51–Q60)

Complex multi-handler chains, polymorphic relationship resolution, and what-if scenarios.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../.."))
from pcg_example.benchmark.notebook_helpers import (
    get_session, run_sql, explode_bom, bom_to_df,
    build_transport_graph, resolve_location_key, display_path
)
import networkx as nx
import pandas as pd
conn, ontology = get_session()
G = build_transport_graph(conn)

## Q51

**"For SKU-ORAL-001, give me the grand unified sourcing view: every raw ingredient, which suppliers provide it, the cheapest inbound shipping route from each supplier to the Dallas plant (PLANT-TX), and the outbound route from Dallas to each retail location that orders this SKU. End-to-end."**

In [ ]:
# Step 1: BOM explosion
bom = explode_bom(conn, sku_code='SKU-ORAL-001', resolve_costs=True)
bom_df = bom_to_df(bom)
print("Step 1 — BOM ingredients:")
display(bom_df[['ingredient_code', 'ingredient_name', 'cumulative_quantity_kg', 'cheapest_supplier', 'cheapest_unit_cost']])

# Step 2: Inbound routes from cheapest suppliers to PLANT-TX
plant_tx = resolve_location_key(conn, 'PLANT-TX')
supplier_routes = []
for _, row in bom_df.drop_duplicates('cheapest_supplier_code').iterrows():
    if row.get('cheapest_supplier_code'):
        sup_key = f"supplier:{run_sql(conn, 'SELECT id FROM suppliers WHERE supplier_code = %s', (row['cheapest_supplier_code'],)).iloc[0,0]}"
        try:
            dist = nx.shortest_path_length(G, sup_key, plant_tx, weight='distance_km')
            supplier_routes.append({'supplier': row['cheapest_supplier_code'], 'distance_km': round(dist, 1)})
        except nx.NetworkXNoPath:
            supplier_routes.append({'supplier': row['cheapest_supplier_code'], 'distance_km': None})
print("\nStep 2 — Inbound supplier routes to PLANT-TX:")
display(pd.DataFrame(supplier_routes))

# Step 3: Outbound routes from PLANT-TX to retail locations that order this SKU
retail_locs = run_sql(conn, """
    SELECT DISTINCT rl.id, rl.location_code
    FROM orders o
    JOIN order_lines ol ON ol.order_id = o.id
    JOIN skus s ON ol.sku_id = s.id
    JOIN retail_locations rl ON o.retail_location_id = rl.id
    WHERE s.sku_code = 'SKU-ORAL-001'
    LIMIT 10
""")
outbound = []
for _, row in retail_locs.iterrows():
    store_key = f"store:{row['id']}"
    try:
        dist = nx.shortest_path_length(G, plant_tx, store_key, weight='distance_km')
        outbound.append({'retail_location': row['location_code'], 'distance_km': round(dist, 1)})
    except nx.NetworkXNoPath:
        outbound.append({'retail_location': row['location_code'], 'distance_km': None})
print(f"\nStep 3 — Outbound routes from PLANT-TX to ordering retail locations (first 10):")
display(pd.DataFrame(outbound))

## Q52

**"A customer filed a quality complaint about order ORD-100-RET-DC-001-42. Trace backward: which production batch filled that order, what formula was used, what ingredients were consumed, and which suppliers provided them? I need the full genealogy."**

In [ ]:
# Trace: order -> shipment -> batch -> formula -> ingredients -> suppliers
run_sql(conn, """
    SELECT o.order_number,
           sh.shipment_number,
           sl.sku_id,
           s.sku_code,
           b.batch_number, b.production_date,
           f.formula_code,
           i.ingredient_code, i.name as ingredient_name,
           bting.quantity_kg as consumed_kg,
           sup.supplier_code, sup.name as supplier_name
    FROM orders o
    JOIN shipments sh ON sh.order_id = o.id
    JOIN shipment_lines sl ON sl.shipment_id = sh.id
    JOIN skus s ON sl.sku_id = s.id
    JOIN batches b ON b.product_id = s.id AND b.product_type = 'finished_good'
    JOIN formulas f ON b.formula_id = f.id
    LEFT JOIN batch_ingredients bting ON bting.batch_id = b.id
    LEFT JOIN ingredients i ON bting.ingredient_id = i.id
    LEFT JOIN supplier_ingredients si ON si.ingredient_id = i.id
    LEFT JOIN suppliers sup ON si.supplier_id = sup.id
    WHERE o.order_number = 'ORD-100-RET-DC-001-42'
    ORDER BY b.batch_number, i.ingredient_code, sup.supplier_code
""")

## Q53

**"What is the fastest route from any manufacturing plant to retail location STORE-PHARM-001-0200, weighted by transit time? The origin type is \"plant\" but I don't care which one — find the best."**

In [ ]:
dst = resolve_location_key(conn, 'STORE-PHARM-001-0200')
results = []
for code in ['PLANT-TX', 'PLANT-OH', 'PLANT-CA', 'PLANT-GA']:
    src = resolve_location_key(conn, code)
    try:
        time = nx.shortest_path_length(G, src, dst, weight='transit_time_hours')
        path = nx.shortest_path(G, src, dst, weight='transit_time_hours')
        results.append({'plant': code, 'transit_hours': round(time, 2), 'hops': len(path)-1, 'path': path})
    except nx.NetworkXNoPath:
        results.append({'plant': code, 'transit_hours': None, 'hops': None, 'path': None})

df = pd.DataFrame([{k: v for k, v in r.items() if k != 'path'} for r in results]).sort_values('transit_hours')
display(df)

best = min((r for r in results if r['transit_hours']), key=lambda r: r['transit_hours'])
print(f"\nFastest: {best['plant']} at {best['transit_hours']:.2f} hours")
display(display_path(G, best['path'], 'transit_time_hours'))

## Q54

**"Show me all inventory currently sitting at distribution center locations only — not plants, not retail. Use the location type to filter. Include the SKU, quantity, and which DC."**

In [ ]:
run_sql(conn, """
    SELECT dc.dc_code, dc.type as dc_type, dc.name as dc_name,
           s.sku_code, s.name as sku_name,
           inv.quantity_cases
    FROM inventory inv
    JOIN distribution_centers dc ON inv.location_id = dc.id
    JOIN skus s ON inv.sku_id = s.id
    WHERE inv.location_type IN ('rdc', 'customer_dc')
      AND inv.day = (SELECT MAX(day) FROM inventory)
      AND inv.quantity_cases > 0
    ORDER BY dc.dc_code, s.sku_code
""")

## Q55

**"Which raw material suppliers ultimately feed products that are shipped to retail locations in the Atlanta metro area? Trace from supplier through BOM through production through fulfillment."**

In [ ]:
# Atlanta metro retail locations -> orders -> SKUs -> formulas -> ingredients -> suppliers
run_sql(conn, """
    SELECT DISTINCT sup.supplier_code, sup.name as supplier_name,
           i.ingredient_code, s.sku_code
    FROM retail_locations rl
    JOIN orders o ON o.retail_location_id = rl.id
    JOIN order_lines ol ON ol.order_id = o.id
    JOIN skus s ON ol.sku_id = s.id
    JOIN formulas f ON f.product_id = s.id AND f.bom_level = 0
    JOIN formula_ingredients fi ON fi.formula_id = f.id
    JOIN ingredients i ON fi.ingredient_id = i.id
    JOIN supplier_ingredients si ON si.ingredient_id = i.id
    JOIN suppliers sup ON si.supplier_id = sup.id
    WHERE rl.city = 'Atlanta'
    ORDER BY sup.supplier_code, i.ingredient_code
    LIMIT 50
""")

## Q56

**"Are there any circular references in our formula hierarchy? Can a formula's ingredient list eventually loop back to itself through bulk intermediates or premix sub-intermediates? We need to rule out BOM cycles before the MRP run."**

In [ ]:
# Cycle detection using recursive CTE with path tracking
df = run_sql(conn, """
    WITH RECURSIVE bom_chain AS (
        -- Base: all formula -> ingredient edges where ingredient is a bulk intermediate
        SELECT f.id as formula_id, f.formula_code, fi.ingredient_id,
               bi.bulk_code, ARRAY[f.id] as path, false as is_cycle
        FROM formulas f
        JOIN formula_ingredients fi ON fi.formula_id = f.id
        JOIN bulk_intermediates bi ON fi.ingredient_id = bi.id
        
        UNION ALL
        
        -- Recurse: follow bulk intermediate -> its formula -> its ingredients
        SELECT f2.id, f2.formula_code, fi2.ingredient_id,
               bi2.bulk_code, bc.path || f2.id,
               f2.id = ANY(bc.path)
        FROM bom_chain bc
        JOIN formulas f2 ON f2.product_id = bc.ingredient_id
        JOIN formula_ingredients fi2 ON fi2.formula_id = f2.id
        JOIN bulk_intermediates bi2 ON fi2.ingredient_id = bi2.id
        WHERE NOT f2.id = ANY(bc.path)
          AND array_length(bc.path, 1) < 10
    )
    SELECT * FROM bom_chain WHERE is_cycle = true
""")
if len(df) == 0:
    print("No circular references found in formula hierarchy. BOM is acyclic.")
else:
    print("WARNING: Circular references detected!")
    display(df)

## Q57

**"For batch B-002-000328, compare the planned ingredient quantities from the formula against the actual quantities consumed from batch_ingredients. Show each ingredient and the delta in kg. Where did we deviate from the recipe?"**

In [ ]:
run_sql(conn, """
    SELECT i.ingredient_code, i.name as ingredient_name,
           fi.quantity_kg as planned_kg,
           bting.quantity_kg as actual_kg,
           (bting.quantity_kg - fi.quantity_kg) as delta_kg,
           ROUND(100.0 * (bting.quantity_kg - fi.quantity_kg) / NULLIF(fi.quantity_kg, 0), 2) as delta_pct
    FROM batches b
    JOIN formula_ingredients fi ON fi.formula_id = b.formula_id
    LEFT JOIN batch_ingredients bting ON bting.batch_id = b.id AND bting.ingredient_id = fi.ingredient_id
    JOIN ingredients i ON fi.ingredient_id = i.id
    WHERE b.batch_number = 'B-002-000328'
    ORDER BY ABS(bting.quantity_kg - fi.quantity_kg) DESC
""")

## Q58

**"If supplier SUP-003 (Apex Ingredients LLC, and their \"-ALT\" duplicate SUP-003-ALT) goes offline tomorrow, which finished SKUs are affected, which open orders contain those SKUs, and which pending shipments are at risk? Quantify the blast radius by revenue exposure."**

In [ ]:
# Step 1: Find affected ingredients
ingredients_df = run_sql(conn, """
    SELECT DISTINCT i.id as ingredient_id, i.ingredient_code, i.name
    FROM suppliers s
    JOIN supplier_ingredients si ON si.supplier_id = s.id
    JOIN ingredients i ON si.ingredient_id = i.id
    WHERE s.supplier_code IN ('SUP-003', 'SUP-003-ALT')
""")
print(f"Step 1 — Affected ingredients: {len(ingredients_df)}")
display(ingredients_df)

# Step 2: Affected formulas and SKUs
ingredient_ids = list(ingredients_df['ingredient_id'])
ph = ','.join(['%s'] * len(ingredient_ids))
affected_skus = run_sql(conn, f"""
    SELECT DISTINCT s.id as sku_id, s.sku_code, s.name as sku_name
    FROM formula_ingredients fi
    JOIN formulas f ON fi.formula_id = f.id AND f.bom_level = 0
    JOIN skus s ON f.product_id = s.id
    WHERE fi.ingredient_id IN ({ph})
""", ingredient_ids)
print(f"\nStep 2 — Affected SKUs: {len(affected_skus)}")
display(affected_skus)

# Step 3: Open orders with those SKUs
sku_ids = list(affected_skus['sku_id'])
ph2 = ','.join(['%s'] * len(sku_ids))
affected_orders = run_sql(conn, f"""
    SELECT o.order_number, o.status, ol.quantity_cases, s.sku_code
    FROM orders o
    JOIN order_lines ol ON ol.order_id = o.id
    JOIN skus s ON ol.sku_id = s.id
    WHERE ol.sku_id IN ({ph2})
      AND o.status IN ('pending', 'allocated')
    ORDER BY o.status, o.order_number
""", sku_ids)
print(f"\nStep 3 — Open/allocated orders: {len(affected_orders)}")
display(affected_orders.head(20))

# Step 4: Revenue exposure
revenue = run_sql(conn, f"""
    SELECT SUM(arl.line_amount) as total_revenue_exposure
    FROM ar_invoice_lines arl
    WHERE arl.sku_id IN ({ph2})
""", sku_ids)
print(f"\nStep 4 — Total AR revenue exposure: ${revenue.iloc[0,0]:,.2f}")

## Q59

**"If the Dallas plant (PLANT-TX) shuts down for two weeks, which orders in \"pending\" or \"allocated\" status cannot be fulfilled from current inventory? Show the order numbers, SKUs, and the gap in cases."**

In [ ]:
# Find SKUs produced at PLANT-TX
tx_skus = run_sql(conn, """
    SELECT DISTINCT s.id as sku_id, s.sku_code
    FROM batches b
    JOIN skus s ON b.product_id = s.id AND b.product_type = 'finished_good'
    WHERE b.plant_id = (SELECT id FROM plants WHERE plant_code = 'PLANT-TX')
""")
sku_ids = list(tx_skus['sku_id'])
ph = ','.join(['%s'] * len(sku_ids))

# Pending/allocated orders for those SKUs
orders = run_sql(conn, f"""
    SELECT o.order_number, o.status, ol.quantity_cases as ordered_cases, s.sku_code
    FROM orders o
    JOIN order_lines ol ON ol.order_id = o.id
    JOIN skus s ON ol.sku_id = s.id
    WHERE ol.sku_id IN ({ph})
      AND o.status IN ('pending', 'allocated')
    ORDER BY s.sku_code, o.order_number
""", sku_ids)

# Current inventory
inventory = run_sql(conn, f"""
    SELECT inv.sku_id, s.sku_code, SUM(inv.quantity_cases) as available_cases
    FROM inventory inv
    JOIN skus s ON inv.sku_id = s.id
    WHERE inv.sku_id IN ({ph})
      AND inv.day = (SELECT MAX(day) FROM inventory)
    GROUP BY inv.sku_id, s.sku_code
""", sku_ids)

# Calculate gap per SKU
demand = orders.groupby('sku_code')['ordered_cases'].sum().reset_index()
demand.columns = ['sku_code', 'total_demand']
supply = inventory[['sku_code', 'available_cases']]
gap = demand.merge(supply, on='sku_code', how='left').fillna(0)
gap['gap_cases'] = gap['total_demand'] - gap['available_cases']
gap = gap[gap['gap_cases'] > 0].sort_values('gap_cases', ascending=False)
print(f"SKUs that cannot be fulfilled from inventory if PLANT-TX shuts down:")
display(gap)
print(f"\nTotal orders at risk: {len(orders)}")

## Q60

**"Show me all shipments that originated from our plants — not DCs — even though the shipment table doesn't have a clean origin type column. You'll need to use the route_type column to figure out which origin IDs are plants."**

In [ ]:
# route_type encodes origin->destination pattern
# Plant origins typically have route_type starting with 'plant_to_'
run_sql(conn, """
    SELECT sh.shipment_number, sh.ship_date, sh.arrival_date,
           sh.route_type, sh.origin_id,
           p.plant_code, p.name as plant_name,
           sh.freight_cost, sh.status
    FROM shipments sh
    JOIN plants p ON sh.origin_id = p.id
    WHERE sh.route_type LIKE 'plant_%'
    ORDER BY sh.ship_date
    LIMIT 30
""")

In [ ]:
conn.close()
print("Session closed.")